<a href="https://colab.research.google.com/github/waghmodedevidas121-cloud/PARAM/blob/main/SadTalker_LatentSync_Colab_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎙️ SadTalker + LatentSync 1.5 — T4 Gradio Studio

**One portrait + speech → natural head/eye/expression motion → refined high-accuracy lips.**

```text
Photo + Audio → SadTalker → motion video → LatentSync 1.5 → final MP4
```

- SadTalker creates head pose, eye blink, expression, and talking motion from a still photo.
- LatentSync 1.5 performs the final lip/teeth synchronization pass.
- LatentSync-only and SadTalker-only modes are included.
- T4 compatible: LatentSync 1.5 requires about 8 GB VRAM; v1.6 requires about 18 GB and is deliberately excluded.
- Both projects are pinned; no API key or paid service is used.

> Use only faces, videos, and voices you own or have permission to animate. Disclose synthetic media.


## Runtime notes

- Target: Colab T4 15 GB.
- Model download: roughly 7–9 GB plus two isolated Python environments.
- Start with 3–8 seconds of speech.
- Combined mode is slower than SadTalker alone but far faster than OmniAvatar because it edits only the face/mouth rather than generating every video pixel.
- “Full image” preserves the source body/background, but SadTalker still animates primarily the head region—it does not generate new hand gestures.


## 0 · GPU and disk check


In [ ]:
import shutil, subprocess
r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],text=True,capture_output=True)
if r.returncode: raise RuntimeError("Select Runtime → Change runtime type → T4 GPU")
print("GPU:",r.stdout.strip())
free=shutil.disk_usage('/content').free/1024**3
print(f"Free disk: {free:.1f} GiB")
if free<18: raise RuntimeError("At least 18 GiB free disk is recommended")


## 1 · Clone pinned source repositories


In [ ]:
import shutil, subprocess
from pathlib import Path
repos=[
    (Path('/content/LatentSync'),'https://github.com/bytedance/LatentSync.git','a229c3948406bc2cf6eaf4873e662e70c6a04746'),
    (Path('/content/SadTalker'),'https://github.com/OpenTalker/SadTalker.git','cd4c0465ae0b54a6f85af57f5c65fec9fe23e7f8'),
]
for root,url,commit in repos:
    if root.exists() and not (root/'.git').exists(): shutil.rmtree(root)
    if not (root/'.git').exists():
        subprocess.run(['git','clone','--filter=blob:none','--no-checkout',url,str(root)],check=True)
    subprocess.run(['git','-C',str(root),'fetch','--depth','1','origin',commit],check=True)
    subprocess.run(['git','-C',str(root),'checkout','--detach',commit],check=True)
    print(root,subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())


## 2 · Install two isolated environments

Separate environments avoid SadTalker's legacy Torch/Numpy stack conflicting with LatentSync's newer diffusion stack.


In [ ]:
import subprocess, sys
from pathlib import Path

def sh(cmd):
    print('+',cmd); subprocess.run(['bash','-lc',cmd],check=True)
sh("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1 libgl1 build-essential python3-dev fonts-dejavu-core")
sh(f"{sys.executable} -m pip install -q 'uv>=0.8,<1'")

SAD=Path('/content/sadtalker-env'); LAT=Path('/content/latentsync-env')
sh(f"{sys.executable} -m uv venv --python 3.10 --clear {SAD}")
sh(f"{sys.executable} -m uv pip install --python {SAD}/bin/python torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118")
sad_pkgs=[
 'numpy==1.23.5','scipy==1.10.1','librosa==0.10.2.post1','numba==0.57.1','llvmlite==0.40.1',
 'face-alignment==1.3.5','imageio==2.31.1','imageio-ffmpeg==0.4.9','resampy==0.3.1',
 'pydub==0.25.1','kornia==0.6.8','tqdm','yacs==0.1.8','PyYAML','joblib',
 'scikit-image==0.21.0','basicsr==1.4.2','facexlib==0.3.0','gfpgan==1.3.8',
 'av','safetensors','opencv-python-headless==4.8.1.78','Pillow==9.5.0','soundfile','cython==3.0.11','wheel'
]
sh(f"{sys.executable} -m uv pip install --python {SAD}/bin/python "+' '.join(repr(x) for x in sad_pkgs))

sh(f"{sys.executable} -m uv venv --python 3.10 --clear {LAT}")
sh(f"{sys.executable} -m uv pip install --python {LAT}/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
lat_pkgs=[
 'diffusers==0.32.2','transformers==4.48.0','decord==0.6.0','accelerate==0.26.1',
 'einops==0.7.0','omegaconf==2.3.0','opencv-python==4.9.0.80','mediapipe==0.10.11',
 'python_speech_features==0.6','librosa==0.10.1','scenedetect==0.6.1','ffmpeg-python==0.2.0',
 'imageio==2.31.1','imageio-ffmpeg==0.5.1','lpips==0.1.4','face-alignment==1.4.1',
 'gradio==5.24.0','huggingface-hub==0.30.2','hf-xet>=1.1.5,<2','numpy==1.26.4',
 'kornia==0.8.0','insightface==0.7.3','onnxruntime-gpu==1.21.0','DeepCache==0.1.1',
 'soundfile','Pillow==11.0.0','setuptools==75.1.0','cython==3.0.11','wheel','ninja'
]
sh(f"{sys.executable} -m uv pip install --python {LAT}/bin/python "+' '.join(repr(x) for x in lat_pkgs))
print('Both environments are ready')


## 3 · Apply T4 and offline-model patches


In [ ]:
from pathlib import Path
p=Path('/content/LatentSync/scripts/inference.py')
s=p.read_text()
s=s.replace('torch.cuda.get_device_capability()[0] > 7','torch.cuda.get_device_capability()[0] >= 7')
s=s.replace('AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=dtype)','AutoencoderKL.from_pretrained("checkpoints/sd-vae-ft-mse", torch_dtype=dtype, local_files_only=True)')
p.write_text(s)
print('Patched LatentSync T4 FP16 + local VAE loading')


## 4 · Download pinned checkpoints


In [ ]:
import subprocess, textwrap
PY='/content/latentsync-env/bin/python'
script=r"""
from pathlib import Path
from huggingface_hub import snapshot_download
import shutil

lat=Path('/content/LatentSync/checkpoints')
snapshot_download('ByteDance/LatentSync-1.5',revision='32a20d29aead0498e3e885e90dbbe8027da1b61b',local_dir=lat,allow_patterns=['latentsync_unet.pt','whisper/tiny.pt','auxiliary/s3fd-619a316812.pth'],max_workers=2)
snapshot_download('stabilityai/sd-vae-ft-mse',revision='31f26fdeee1355a5c34592e401dd41e45d25a493',local_dir=lat/'sd-vae-ft-mse',allow_patterns=['config.json','diffusion_pytorch_model.safetensors'],max_workers=2)

tmp=Path('/content/sadtalker-models')
snapshot_download('camenduru/SadTalker',revision='7caf8d94b767aa60b5a6768ada79fde0c6713b76',local_dir=tmp,allow_patterns=[
 'new/checkpoints/SadTalker_V0.0.2_256.safetensors','new/checkpoints/mapping_00109-model.pth.tar','new/checkpoints/mapping_00229-model.pth.tar',
 'new/gfpgan/weights/alignment_WFLW_4HG.pth','new/gfpgan/weights/detection_Resnet50_Final.pth'],max_workers=2)
sad=Path('/content/SadTalker'); (sad/'checkpoints').mkdir(exist_ok=True); (sad/'gfpgan/weights').mkdir(parents=True,exist_ok=True)
for f in (tmp/'new/checkpoints').glob('*'): shutil.copy2(f,sad/'checkpoints'/f.name)
for f in (tmp/'new/gfpgan/weights').glob('*'): shutil.copy2(f,sad/'gfpgan/weights'/f.name)
# Seed the torch-hub S3FD cache used by face-alignment when required.
hub=Path.home()/'.cache/torch/hub/checkpoints'; hub.mkdir(parents=True,exist_ok=True)
s3fd=lat/'auxiliary/s3fd-619a316812.pth'
if s3fd.exists(): shutil.copy2(s3fd,hub/s3fd.name)
required=[lat/'latentsync_unet.pt',lat/'whisper/tiny.pt',sad/'checkpoints/SadTalker_V0.0.2_256.safetensors',sad/'gfpgan/weights/alignment_WFLW_4HG.pth']
missing=[str(x) for x in required if not x.is_file()]
if missing: raise RuntimeError('Missing: '+', '.join(missing))
print('All checkpoints ready')
"""
subprocess.run([PY,'-c',textwrap.dedent(script)],check=True)


## 5 · Download InsightFace detector once and verify environments


In [ ]:
import os, subprocess, textwrap
LAT='/content/latentsync-env/bin/python'; SAD='/content/sadtalker-env/bin/python'
env=os.environ.copy(); env['MPLBACKEND']='Agg'
checks=[
 [SAD,'-c',"import torch,cv2,librosa; print('SadTalker env',torch.__version__,torch.cuda.is_available())"],
 [LAT,'-c',"import torch,diffusers,insightface,gradio; print('LatentSync env',torch.__version__,torch.cuda.is_available(),diffusers.__version__)"],
]
for cmd in checks:
 r=subprocess.run(cmd,text=True,capture_output=True,env=env); print(r.stdout,r.stderr); assert r.returncode==0
bootstrap=r"""
from insightface.app import FaceAnalysis
app=FaceAnalysis(allowed_modules=['detection','landmark_2d_106'],root='checkpoints/auxiliary',providers=['CUDAExecutionProvider','CPUExecutionProvider'])
app.prepare(ctx_id=0,det_size=(512,512))
print('InsightFace models ready')
"""
r=subprocess.run([LAT,'-c',textwrap.dedent(bootstrap)],cwd='/content/LatentSync',text=True,capture_output=True,env=env)
print(r.stdout,r.stderr)
if r.returncode: raise RuntimeError('InsightFace bootstrap failed')


## 6 · Create combined Gradio app


In [ ]:
from pathlib import Path
ROOT=Path('/content/AvatarSync'); ROOT.mkdir(exist_ok=True)
APP_PATH=ROOT/'sadtalker_latentsync_gradio.py'
APP_CODE='from __future__ import annotations\n\nimport argparse\nimport gc\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom pathlib import Path\n\nos.environ["MPLBACKEND"] = "Agg"\nos.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nprint("[startup 1/3] Importing Gradio and media helpers…", flush=True)\nimport gradio as gr\nimport soundfile as sf\nfrom PIL import Image, ImageOps\n\nAPP_ROOT = Path(__file__).resolve().parent\nSAD_ROOT = Path("/content/SadTalker")\nLATENT_ROOT = Path("/content/LatentSync")\nSAD_PYTHON = Path("/content/sadtalker-env/bin/python")\nLATENT_PYTHON = Path("/content/latentsync-env/bin/python")\nOUTPUT_ROOT = APP_ROOT / "outputs"\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n\nFFMPEG_BIN = shutil.which("ffmpeg")\nif FFMPEG_BIN is None:\n    from imageio_ffmpeg import get_ffmpeg_exe\n    FFMPEG_BIN = get_ffmpeg_exe()\n\nMODEL_LOCK = threading.Lock()\nMAX_AUDIO_SECONDS = 30.0\n\n\ndef clean_old_jobs(max_age_hours: float = 10.0) -> None:\n    cutoff = time.time() - max_age_hours * 3600\n    for item in OUTPUT_ROOT.iterdir():\n        try:\n            if item.is_dir() and item.stat().st_mtime < cutoff:\n                shutil.rmtree(item, ignore_errors=True)\n        except OSError:\n            pass\n\n\ndef run_checked(command: list[str]) -> None:\n    result = subprocess.run(command, text=True, capture_output=True)\n    if result.returncode:\n        raise RuntimeError((result.stderr or result.stdout)[-3000:])\n\n\ndef prepare_image(source: str, destination: Path) -> None:\n    Image.MAX_IMAGE_PIXELS = 40_000_000\n    with Image.open(source) as opened:\n        image = ImageOps.exif_transpose(opened).convert("RGB")\n        width, height = image.size\n        if min(width, height) < 256:\n            raise ValueError("Image is too small. Use at least 256 px on each side.")\n        image.save(destination, "PNG", optimize=True)\n\n\ndef prepare_audio(source: str, destination: Path) -> float:\n    run_checked([\n        FFMPEG_BIN, "-hide_banner", "-loglevel", "error", "-y",\n        "-i", source, "-vn", "-ac", "1", "-ar", "16000",\n        "-c:a", "pcm_s16le", str(destination),\n    ])\n    info = sf.info(str(destination))\n    duration = float(info.frames / info.samplerate)\n    if duration < 0.5:\n        raise ValueError("Audio is too short. Use at least 0.5 seconds.")\n    if duration > MAX_AUDIO_SECONDS:\n        raise ValueError(f"Audio is {duration:.1f}s; trim it to {MAX_AUDIO_SECONDS:.0f}s or less.")\n    return duration\n\n\ndef prepare_video(source: str, destination: Path) -> None:\n    run_checked([\n        FFMPEG_BIN, "-hide_banner", "-loglevel", "error", "-y",\n        "-i", source, "-an", "-vf", "fps=25",\n        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",\n        "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(destination),\n    ])\n\n\ndef stream_process(command: list[str], cwd: Path, env: dict[str, str], log_file) -> int:\n    print("\\n[run]", " ".join(command), flush=True)\n    process = subprocess.Popen(\n        command, cwd=cwd, env=env, stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT, text=True, bufsize=1,\n    )\n    assert process.stdout is not None\n    for line in process.stdout:\n        print(line, end="", flush=True)\n        log_file.write(line)\n        log_file.flush()\n    return process.wait()\n\n\ndef add_watermark(source: Path, destination: Path) -> bool:\n    font = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"\n    vf = (\n        "drawbox=x=w-238:y=h-52:w=226:h=40:color=black@0.55:t=fill,"\n        f"drawtext=fontfile={font}:text=\'AI-generated avatar\':"\n        "fontcolor=white:fontsize=18:x=w-tw-20:y=h-th-20"\n    )\n    result = subprocess.run([\n        FFMPEG_BIN, "-hide_banner", "-loglevel", "error", "-y",\n        "-i", str(source), "-vf", vf, "-c:v", "libx264",\n        "-preset", "veryfast", "-crf", "18", "-c:a", "copy",\n        "-movflags", "+faststart", str(destination),\n    ], text=True, capture_output=True)\n    return result.returncode == 0\n\n\ndef generate(\n    mode: str,\n    image: str,\n    source_video: str,\n    audio: str,\n    preprocess: str,\n    pose_style: int,\n    expression_scale: float,\n    stable_head: bool,\n    latent_steps: int,\n    latent_guidance: float,\n    deepcache: bool,\n    seed: int,\n    disclosure: bool,\n    consent: bool,\n    progress=gr.Progress(),\n):\n    if not consent:\n        raise gr.Error("Confirm that you have permission to use the face, video, and audio.")\n    if not audio:\n        raise gr.Error("Upload or record driving speech audio.")\n    if mode != "LatentSync only (video + audio)" and not image:\n        raise gr.Error("Upload a source portrait for SadTalker.")\n    if mode == "LatentSync only (video + audio)" and not source_video:\n        raise gr.Error("Upload an existing source video for LatentSync-only mode.")\n\n    with MODEL_LOCK:\n        clean_old_jobs()\n        started = time.perf_counter()\n        job = OUTPUT_ROOT / f"job_{time.strftime(\'%Y%m%d_%H%M%S\')}_{uuid.uuid4().hex[:8]}"\n        job.mkdir(parents=True, exist_ok=False)\n        input_image = job / "source.png"\n        input_audio = job / "speech.wav"\n        input_video = job / "source_video.mp4"\n        sad_dir = job / "sadtalker_results"\n        sad_video = job / "sadtalker.mp4"\n        synced_video = job / "latentsync.mp4"\n        final_video = job / "avatar.mp4"\n        log_path = job / "pipeline.log"\n        sad_seconds = 0.0\n        latent_seconds = 0.0\n\n        env = os.environ.copy()\n        env.update({\n            "CUDA_VISIBLE_DEVICES": "0",\n            "MPLBACKEND": "Agg",\n            "PYTHONUNBUFFERED": "1",\n            "TOKENIZERS_PARALLELISM": "false",\n        })\n        # Torch 2.0 in SadTalker rejects the newer expandable_segments option.\n        env.pop("PYTORCH_CUDA_ALLOC_CONF", None)\n\n        try:\n            progress(0.03, desc="Preparing inputs…")\n            duration = prepare_audio(audio, input_audio)\n            if image:\n                prepare_image(image, input_image)\n            if source_video:\n                prepare_video(source_video, input_video)\n\n            with log_path.open("w", encoding="utf-8") as log:\n                if mode != "LatentSync only (video + audio)":\n                    progress(0.08, desc="SadTalker: generating head, eye, and expression motion…")\n                    sad_dir.mkdir(parents=True, exist_ok=True)\n                    sad_command = [\n                        str(SAD_PYTHON), "inference.py",\n                        "--driven_audio", str(input_audio),\n                        "--source_image", str(input_image),\n                        "--checkpoint_dir", "checkpoints",\n                        "--result_dir", str(sad_dir),\n                        "--preprocess", preprocess,\n                        "--size", "256",\n                        "--pose_style", str(int(pose_style)),\n                        "--expression_scale", str(float(expression_scale)),\n                        "--batch_size", "2",\n                    ]\n                    if stable_head:\n                        sad_command.append("--still")\n                    stage_start = time.perf_counter()\n                    code = stream_process(sad_command, SAD_ROOT, env, log)\n                    sad_seconds = time.perf_counter() - stage_start\n                    if code:\n                        raise RuntimeError(f"SadTalker exited with code {code}")\n                    candidates = sorted(sad_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)\n                    if not candidates:\n                        raise RuntimeError("SadTalker completed but no MP4 was found.")\n                    shutil.copy2(candidates[-1], sad_video)\n                    latent_input = sad_video\n                else:\n                    latent_input = input_video\n\n                if mode != "SadTalker only (fast draft)":\n                    progress(0.48, desc="LatentSync 1.5: refining final lip synchronization…")\n                    temp_dir = job / "latentsync_temp"\n                    latent_command = [\n                        str(LATENT_PYTHON), "-m", "scripts.inference",\n                        "--unet_config_path", "configs/unet/stage2.yaml",\n                        "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",\n                        "--video_path", str(latent_input),\n                        "--audio_path", str(input_audio),\n                        "--video_out_path", str(synced_video),\n                        "--inference_steps", str(int(latent_steps)),\n                        "--guidance_scale", str(float(latent_guidance)),\n                        "--seed", str(int(seed)),\n                        "--temp_dir", str(temp_dir),\n                    ]\n                    if deepcache:\n                        latent_command.append("--enable_deepcache")\n                    latent_env = env.copy()\n                    # Supported by LatentSync\'s Torch 2.5, but not SadTalker\'s Torch 2.0.\n                    latent_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\n                    stage_start = time.perf_counter()\n                    code = stream_process(latent_command, LATENT_ROOT, latent_env, log)\n                    latent_seconds = time.perf_counter() - stage_start\n                    if code:\n                        raise RuntimeError(f"LatentSync exited with code {code}")\n                    if not synced_video.is_file():\n                        raise RuntimeError("LatentSync completed but no MP4 was found.")\n                    raw_output = synced_video\n                else:\n                    raw_output = sad_video\n\n            progress(0.95, desc="Finalizing video…")\n            watermarked = disclosure and add_watermark(raw_output, final_video)\n            selected = final_video if watermarked else raw_output\n            elapsed = time.perf_counter() - started\n            status = (\n                "### ✅ Avatar ready\\n"\n                f"- Mode: **{mode}** · Audio: **{duration:.1f}s**\\n"\n                f"- SadTalker: **{sad_seconds:.1f}s** · LatentSync: **{latent_seconds:.1f}s**\\n"\n                f"- Total: **{elapsed:.1f}s** · Latent steps: **{int(latent_steps)}**"\n            )\n            if disclosure and not watermarked:\n                status += "\\n- ⚠️ Disclosure watermark could not be added; raw output is shown."\n            gc.collect()\n            progress(1.0, desc="Done")\n            return str(selected), status, str(log_path)\n        except gr.Error:\n            raise\n        except Exception as exc:\n            traceback.print_exc()\n            raise gr.Error(f"Pipeline failed: {exc}. Download/check the pipeline log if available.") from exc\n\n\nCSS = """\n.gradio-container {max-width:1220px !important;}\n.hero {text-align:center; padding:12px 0 4px;}\n.hero h1 {font-size:2.2rem; margin-bottom:.25rem;}\n.notice {border-left:4px solid #2563eb; padding:11px 15px; background:rgba(37,99,235,.08); border-radius:8px;}\n"""\n\nprint("[startup 2/3] Building SadTalker + LatentSync interface…", flush=True)\nwith gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="violet"), css=CSS, title="SadTalker + LatentSync") as demo:\n    gr.HTML("""\n    <div class="hero"><h1>🎙️ SadTalker + LatentSync 1.5</h1>\n    <p>Single photo motion generation followed by high-accuracy lip refinement</p></div>\n    """)\n    gr.Markdown(\n        "<div class=\'notice\'><b>Recommended combined mode:</b> SadTalker first creates head, eye, "\n        "and expression motion from one photo; LatentSync 1.5 then regenerates the mouth for stronger "\n        "audio alignment and cleaner temporal lip motion. T4-compatible.</div>"\n    )\n\n    with gr.Row(equal_height=False):\n        with gr.Column(scale=5):\n            mode = gr.Radio(\n                choices=[\n                    "SadTalker + LatentSync (recommended)",\n                    "SadTalker only (fast draft)",\n                    "LatentSync only (video + audio)",\n                ],\n                value="SadTalker + LatentSync (recommended)",\n                label="Pipeline mode",\n            )\n            image = gr.Image(label="Source portrait (required for SadTalker)", type="filepath", sources=["upload", "webcam"], height=360)\n            source_video = gr.Video(label="Existing video (only for LatentSync-only mode)", sources=["upload"], height=220)\n            audio = gr.Audio(label="Driving speech (max 30 seconds)", type="filepath", sources=["upload", "microphone"])\n\n            with gr.Accordion("SadTalker motion controls", open=True):\n                preprocess = gr.Radio(\n                    choices=[("Full image / body stays visible", "full"), ("Face crop", "crop")],\n                    value="full", label="Framing",\n                )\n                pose_style = gr.Slider(0, 45, value=0, step=1, label="Head-pose style")\n                expression_scale = gr.Slider(0.5, 1.8, value=1.0, step=0.05, label="Expression scale")\n                stable_head = gr.Checkbox(value=False, label="More stable head (less pose motion)")\n\n            with gr.Accordion("LatentSync quality controls", open=True):\n                latent_steps = gr.Slider(10, 30, value=20, step=2, label="Inference steps")\n                latent_guidance = gr.Slider(1.0, 3.0, value=1.5, step=0.1, label="Guidance scale")\n                deepcache = gr.Checkbox(value=True, label="Enable DeepCache acceleration")\n                seed = gr.Number(value=1247, precision=0, label="Seed")\n                disclosure = gr.Checkbox(value=True, label="Add AI-generated disclosure watermark")\n\n            consent = gr.Checkbox(\n                value=False,\n                label="I own or have permission to use this face/video/audio and will disclose synthetic media.",\n            )\n            generate_button = gr.Button("✨ Generate and refine lips", variant="primary", size="lg")\n\n        with gr.Column(scale=6):\n            output = gr.Video(label="Final video", height=560)\n            status = gr.Markdown("Upload inputs and generate.")\n            pipeline_log = gr.File(label="Pipeline log")\n\n    generate_button.click(\n        fn=generate,\n        inputs=[mode, image, source_video, audio, preprocess, pose_style, expression_scale, stable_head,\n                latent_steps, latent_guidance, deepcache, seed, disclosure, consent],\n        outputs=[output, status, pipeline_log],\n        api_name=False,\n    )\n\n    gr.Markdown(\n        "---\\n**Tips:** start with a front-facing high-resolution portrait and 3–8 seconds of clean speech. "\n        "Full mode preserves the original body/background but only the head region moves. LatentSync 1.5 "\n        "needs about 8 GB VRAM; 1.6 is intentionally excluded because it needs about 18 GB. "\n        "[SadTalker](https://github.com/OpenTalker/SadTalker) · "\n        "[LatentSync](https://github.com/bytedance/LatentSync)"\n    )\n\nprint("[startup 3/3] Interface ready; starting Gradio…", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--host", default="0.0.0.0")\n    parser.add_argument("--port", type=int, default=7862)\n    parser.add_argument("--share", action="store_true")\n    args = parser.parse_args()\n    demo.queue(max_size=3, default_concurrency_limit=1).launch(\n        server_name=args.host, server_port=args.port, share=args.share,\n        show_error=True, allowed_paths=[str(OUTPUT_ROOT)],\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
APP_PATH.write_text(APP_CODE)
print('Wrote',APP_PATH)


## 7 · Launch

Open the printed `gradio.live` URL and keep the cell running.


In [ ]:
import os,subprocess
subprocess.run(['pkill','-f','/content/AvatarSync/sadtalker_latentsync_gradio.py'],check=False)
cmd=['/content/latentsync-env/bin/python','-u','/content/AvatarSync/sadtalker_latentsync_gradio.py','--host','0.0.0.0','--port','7862','--share']
env=os.environ.copy(); env.update({'MPLBACKEND':'Agg','GRADIO_ANALYTICS_ENABLED':'False','PYTHONUNBUFFERED':'1'})
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
try:
 for line in p.stdout: print(line,end='',flush=True)
finally:
 rc=p.wait()
 if rc not in (0,-2,-15): raise RuntimeError(f'Gradio exited: {rc}')


## Recommended first run

- Mode: **SadTalker + LatentSync**
- Portrait: front-facing, shoulders visible
- Audio: 3–6 seconds
- Framing: **Full image** if you want the original body/background visible
- SadTalker pose style: 0; expression: 1.0
- LatentSync: 20 steps, guidance 1.5, DeepCache ON

If combined mode fails, test SadTalker-only and LatentSync-only separately to identify the stage.
